# **📄 Clean & Jerk Dataset — Publication Figures & Tables (`Plan.md`)**

### 🔧 Colab Setup & Library Init

In [13]:
# @title 🔧 Colab Setup & Library Init { display-mode: "form" }
import importlib, sys, types
if "imp" not in sys.modules:
    try:
        import imp  # noqa: F401
    except ModuleNotFoundError:
        _imp = types.ModuleType("imp")
        _imp.reload = importlib.reload
        sys.modules["imp"] = _imp

%load_ext autoreload
%autoreload 2

import importlib.util, os, subprocess

try:
    from google.colab import drive, data_table
    import plotly.io as pio
    plotly_renderer = "vscode" # @param ["vscode", "colab", "notebook_connected", "iframe"]
    pio.renderers.default = plotly_renderer
    data_table.enable_dataframe_formatter()
    if not os.path.exists("/content/drive/MyDrive"):
        try:
            drive.mount('/content/drive', force_remount=False)
        except Exception as e:
            print(f"⚠️ Drive not mounted ({e}). Mount via Colab sidebar if needed.")
except ImportError:
    pass

if importlib.util.find_spec("h5py") is None:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "h5py"])

repo_dir = "/content/Wearable-Biomechanics" # @param {type:"string"}
git_branch = "dataset_explorer" # @param {type:"string"}

if os.path.exists(os.path.join(".", "src", "__init__.py")):
    repo_root = os.path.abspath(".")
else:
    if not os.path.exists(repo_dir):
        subprocess.check_call([
            "git", "clone", "-b", git_branch,
            "https://github.com/janithcyapa/Wearable-Biomechanics.git", repo_dir
        ])
    else:
        subprocess.run(["git", "-C", repo_dir, "checkout", git_branch], check=False)
        subprocess.run(["git", "-C", repo_dir, "pull", "--ff-only"], check=False)
        for _m in [k for k in list(sys.modules) if k == "src" or k.startswith("src.")]:
            importlib.reload(sys.modules[_m])
    repo_root = os.path.abspath(repo_dir)

if repo_root not in sys.path:
    sys.path.insert(0, repo_root)

from src import display_table
from src.explorer import (
    DEFAULT_CLEAN_DATASET_DIR,
    build_annotation_agreement_table,
    build_imu_quality_table,
    build_paper_tables,
    plot_fig4_annotation_scheme,
    plot_all_sets_annotation_sd_heatmap,
    plot_annotation_agreement_figure,
    plot_imu_validation_figure,
    plot_set_annotator_grid,
)

dataset_dir = DEFAULT_CLEAN_DATASET_DIR # @param {type:"string"}
EXPORT_CFG = {"toImageButtonOptions": {"format": "png", "scale": 4}}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


### 📊 Paper Tables

In [3]:
# @title 📊 Generate Paper Tables 5, 7, 8
paper_tables = build_paper_tables(dataset_dir)
for name, df in paper_tables.items():
    print(f"\n=== {name} ===")
    display_table(df, clean_columns=False)


=== table5_demographics ===


   participant_id                       name  age_years sex  height_m  \
0           P0001                      Enuka        NaN   M       NaN   
1           P0002      Thevindu Samarasekera       25.0   M      1.69   
2           P0003             Achalani Royal       23.0   F      1.60   
3           P0004                    Pandula        NaN   M       NaN   
4           P0005          Induwara Liyanage       21.0   M      1.70   
5           P0006                   Duleetha        NaN   M       NaN   
6           P0007         Hivindu Kirulapana       22.0   M      1.74   
7           P0008          Yewin Dissanayake       22.0   M      1.70   
8           P0009         Yasith Dissanayake       22.0   M      1.78   
9           P0010  J W K Mahasen Jayasundara       24.0   M      1.78   
10          P0011      Nirmal Samarawickrama       22.0   M      1.73   
11          P0012   J. M. Sudeshe Jayasinghe       23.0   M      1.56   
12          P0013          Dulan Jayasooriya       


=== table7_coverage ===


                   tier  sets  participants  lifts_total  lifts_usable  \
0  Complete (No Faults)    35            15          157           157   
1        Fault-Tolerant    19             9           89            80   
2                 Total    54            19          246           237   

   duration_min  total_imu_samples  
0         44.37             266229  
1         25.36             152162  
2         69.73             418391  


=== table8_agreement ===


          marker_name  n_boundaries  mean_sd_frames  mean_sd_ms  \
0               start           246            2.64        88.0   
1               setup           246            2.79        93.2   
2          first_pull           246            2.13        70.9   
3         second_pull           246            1.51        50.4   
4               clean           246            2.27        75.5   
5   intermediate_rest           246            3.11       103.5   
6          jerk_split           246            2.27        75.7   
7       jerk_recovery           246            2.85        95.1   
8       overhead_hold           246            3.37       112.3   
9             descend           246            2.51        83.8   
10          post_rest           246            3.38       112.6   
11                end           246            2.05        68.4   

    max_sd_frames  pct_within_6_frames  pct_within_8_frames  
0            7.51                 90.7                100.0  
1   

### 🖼️ 11-Phase Annotation Scheme

In [12]:
# @title 🖼️ Figure 4: Annotation Scheme { display-mode: "form" }
sample_set_id = "set_0052" # @param {type:"string"}
sample_lifts = "3" # @param {type:"string"}  # e.g. "1", "1,2", "1-5", or "all"
sample_sensors = "RHP" # @param {type:"string"}  # e.g. "RHD" or "RKP,RHD" or "US,LS"
modality = "both" # @param ["gyro", "accel", "both"]
signal_mode = "axes" # @param ["axes", "magnitude", "both"]
axes_str = "X,Y,Z" # @param {type:"string"}  # e.g. "X,Y,Z" or "Y" or "X,Z"

fig4 = plot_fig4_annotation_scheme(
    dataset_dir,
    set_id=sample_set_id,
    lift_ids=sample_lifts,
    sensor_codes=sample_sensors,
    modality=modality,
    signal_mode=signal_mode,
    axes=axes_str,
)
fig4.show(config=EXPORT_CFG)

### 📊 IMU Technical Validation Summary Table

In [ ]:
# @title 📊 IMU Quality Summary Table (Option 1)
df_imu_quality = build_imu_quality_table(dataset_dir)
display_table(df_imu_quality, clean_columns=False)

📝 Draft text: "Across all 1,404 sensor recordings (54 sets × 26 IMUs), data completeness was 97.4 ± 15.8% (min 0.0%). The accelerometer norm during static periods was 9.67 ± 1.27 m/s², and static gyroscope bias was 10.29 ± 8.87 deg/s."


,Metric,Expected / Threshold,Mean ± SD,Median [IQR],Min – Max,Recordings Passing (%)
0,Data completeness (% valid samples),> 99%,97.43 ± 15.81,"100.00 [100.00, 100.00]",0.00 – 100.00,97.4%
1,Missing / flatlined samples (%),< 1%,2.57 ± 15.81,"0.00 [0.00, 0.00]",0.00 – 100.00,97.4%
2,Static accelerometer norm (m/s²),9.81 ± 0.50 m/s²,9.67 ± 1.27,"9.81 [9.62, 10.06]",0.00 – 12.39,87.7%
3,Static gyroscope bias magnitude (deg/s),< 5.0 deg/s,10.29 ± 8.87,"7.90 [5.35, 12.13]",0.00 – 125.50,20.8%
4,Accelerometer saturation (% > 80 m/s²),≈ 0% (< 0.1%),0.00 ± 0.00,"0.00 [0.00, 0.00]",0.00 – 0.00,100.0%
5,Gyroscope saturation (% > 2000 deg/s),≈ 0% (< 0.1%),0.00 ± 0.00,"0.00 [0.00, 0.00]",0.00 – 0.00,100.0%
6,Inter-IMU dynamic lag vs. US (ms),< 10 ms (1 sample),104.64 ± 73.98,"100.00 [30.00, 190.00]",0.00 – 200.00,16.1%


### 🖼️ IMU Technical Validation 4-Panel Figure

In [ ]:
# @title 🖼️ IMU Technical Validation Figure (Option 2)
fig_val = plot_imu_validation_figure(dataset_dir)
fig_val.show(config=EXPORT_CFG)

### 🔥 Per-Set Annotator Grid & SD (ms) Heatmap

In [ ]:
# @title 🔥 Per-Set Annotator Grid { display-mode: "form" }
ann_set_id = "set_0001" # @param {type:"string"}
fig_ann_set = plot_set_annotator_grid(dataset_dir, set_id=ann_set_id)
fig_ann_set.show(config=EXPORT_CFG)

### 🔥 All-Sets Inter-Annotator SD (ms) Heatmap

In [ ]:
# @title 🔥 All-Sets Inter-Annotator SD (ms) Heatmap
fig_ann_all = plot_all_sets_annotation_sd_heatmap(dataset_dir)
fig_ann_all.show(config=EXPORT_CFG)

### 📊 Annotation Agreement Summary Table (Right-Skewed)

In [ ]:
# @title 📊 Annotation Agreement Table { display-mode: "form" }
threshold_ms = 200.0 # @param {type:"number"}  # 200 ms = 6 frames @ 30 fps (20 samples @ 100 Hz)
df_ann_table = build_annotation_agreement_table(dataset_dir, threshold_ms=threshold_ms)
display_table(df_ann_table, clean_columns=False)

📝 Draft text: "Of 2,952 marker instances, 100.0% were identified by all three annotators. For these, the inter-annotator SD of marker timestamps was right-skewed, with a median of 69.4 ms [IQR 33.3–126.2 ms]; 90% of instances showed an SD below 176.4 ms (≈ 17.6 samples at 100 Hz). Agreement was highest for first_pull (median 50.9 ms) and lowest for post_rest (median 117.1 ms)."


,Marker type,n instances,Marked by all 3 (%),Median SD (ms) [IQR],Mean ± SD (ms),90th pct (ms),95th pct (ms),% < 200 ms,Max (ms)
0,start,246,100.0%,"69.4 [19.2, 153.7]",88.0 ± 75.9,195.3,216.2,90.7%,250.2
1,setup,246,100.0%,"83.9 [50.9, 134.7]",93.2 ± 56.9,171.1,194.6,95.5%,240.4
2,first_pull,246,100.0%,"50.9 [33.3, 100.0]",70.9 ± 50.5,145.3,170.2,98.4%,250.2
3,second_pull,246,100.0%,"50.9 [33.3, 66.7]",50.4 ± 30.4,88.2,101.4,100.0%,183.6
4,clean,246,100.0%,"69.4 [38.5, 101.8]",75.5 ± 42.5,136.8,153.7,100.0%,189.5
5,intermediate_rest,246,100.0%,"98.1 [50.9, 149.1]",103.5 ± 63.1,200.9,214.3,89.4%,251.7
6,jerk_split,246,100.0%,"66.7 [38.5, 100.0]",75.7 ± 51.0,153.4,192.4,97.6%,236.5
7,jerk_recovery,246,100.0%,"83.9 [50.9, 145.3]",95.1 ± 63.7,187.6,207.0,91.1%,250.2
8,overhead_hold,246,100.0%,"107.2 [52.6, 164.4]",112.3 ± 66.3,202.8,232.3,87.0%,266.7
9,descend,246,100.0%,"69.4 [38.5, 117.1]",83.8 ± 51.9,164.4,183.6,97.2%,214.3


### 🖼️ Annotation Agreement 2-Panel Figure (Histogram + ECDF)

In [ ]:
# @title 🖼️ Annotation Agreement (Histogram + ECDF) { display-mode: "form" }
log_x_ecdf = False # @param {type:"boolean"}
fig_ann_agree = plot_annotation_agreement_figure(dataset_dir, log_x_ecdf=log_x_ecdf)
fig_ann_agree.show(config=EXPORT_CFG)